In [6]:
import os
from pathlib import Path

from sqlmodel import Session


os.environ["DATABASE_URL"] = (
    "postgresql+psycopg://"
    "document_agent:document_agent@127.0.0.1:5432/document_agent"
)

from app.db import engine
from app.rag.ingestion import ingest_document

In [7]:
md_files = list(Path("data/raw").rglob("*.md"))

path = md_files[0]

print(path)

data/raw/pgvector/README.md


In [8]:
with Session(engine) as session:
    document = ingest_document(
        session=session,
        path=path,
        source="fastapi",
        language="en",
        title=path.stem,
    )

    print("Document ID:", document.id)
    print("Title:", document.title)

Document ID: 3
Title: README


In [9]:
from sqlmodel import select
from app.models import DocumentChunk

with Session(engine) as session:
    statement = (
        select(DocumentChunk)
        .where(DocumentChunk.document_id == document.id)
        .order_by(DocumentChunk.chunk_index)
    )

    db_chunks = session.exec(statement).all()

    print("Chunks:", len(db_chunks))

    for chunk in db_chunks[:3]:
        print("=" * 60)
        print("INDEX:", chunk.chunk_index)
        print("HEADING:", chunk.heading_path)
        print("SIZE:", len(chunk.content))
        print(chunk.content[:300])

Chunks: 93
INDEX: 0
HEADING: ['pgvector']
SIZE: 750
Open-source vector similarity search for Postgres

Store your vectors with the rest of your data. Supports:

- exact and approximate nearest neighbor search
- single-precision, half-precision, binary, and sparse vectors
- L2 distance, inner product, cosine distance, L1 distance, Hamming distance, an
INDEX: 1
HEADING: ['pgvector', 'Installation', 'Linux and Mac']
SIZE: 651
Compile and install the extension (supports Postgres 13+)

```sh
cd /tmp
git clone --branch v0.8.6 https://github.com/pgvector/pgvector.git
cd pgvector
make
make install # may need sudo
```

See the [installation notes](#installation-notes---linux-and-mac) if you run into issues

You can also instal
INDEX: 2
HEADING: ['pgvector', 'Installation', 'Windows']
SIZE: 634
Ensure [C++ support in Visual Studio](https://learn.microsoft.com/en-us/cpp/build/building-on-the-command-line?view=msvc-170#download-and-install-the-tools) is installed and run `x64 Native Tools Command 